In [1]:
import tensorflow as tf

# tf dataset from a list

In [2]:
# Create tf dataset from list
daily_sales_num = [21, 22, -108, 31, -1, 32, 34, 31]
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_num)
tf_dataset

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

In [4]:
# Iterate through tf dataset
for sales in tf_dataset:
  print(sales)
  print(sales.numpy())

tf.Tensor(21, shape=(), dtype=int32)
21
tf.Tensor(22, shape=(), dtype=int32)
22
tf.Tensor(-108, shape=(), dtype=int32)
-108
tf.Tensor(31, shape=(), dtype=int32)
31
tf.Tensor(-1, shape=(), dtype=int32)
-1
tf.Tensor(32, shape=(), dtype=int32)
32
tf.Tensor(34, shape=(), dtype=int32)
34
tf.Tensor(31, shape=(), dtype=int32)
31


In [5]:
# Iterate through elements as numpy elements
for sales in tf_dataset.as_numpy_iterator():
  print(sales)

21
22
-108
31
-1
32
34
31


In [7]:
# Iterate through first n elements in tf dataset
for sales in tf_dataset.take(4):
  print(sales.numpy())

21
22
-108
31


In [8]:
# Filter sales numbers that are less than 0
tf_dataset = tf_dataset.filter(lambda x:x>0)
for sales in tf_dataset:
  print(sales.numpy())

21
22
31
32
34
31


In [9]:
# Convert sales numbers from INR to dollars
tf_dataset = tf_dataset.map(lambda x:x*91)
for sales in tf_dataset:
  print(sales.numpy())

1911
2002
2821
2912
3094
2821


In [10]:
# Shuffle
tf_dataset = tf_dataset.shuffle(3)
for sales in tf_dataset:
  print(sales.numpy())

1911
2821
2002
2821
2912
3094


In [11]:
# Batching
for sales_batch in tf_dataset.batch(2):
  print(sales_batch.numpy())

[2002 2912]
[1911 3094]
[2821 2821]


In [13]:
# Perform all of the above operations in one short
tf_dataset = tf.data.Dataset.from_tensor_slices(daily_sales_num)
tf_dataset = tf_dataset.filter(lambda x:x>0).map(lambda y:y*91).shuffle(3).batch(2)

for sales in tf_dataset:
  print(sales.numpy())

[2821 2002]
[2912 3094]
[1911 2821]


# tf dataset from images

In [19]:
# Load images
images_ds = tf.data.Dataset.list_files('images/*/*')
print(images_ds)

<_ShuffleDataset element_spec=TensorSpec(shape=(), dtype=tf.string, name=None)>


In [20]:
# image count
image_count = len(images_ds)
image_count

16

In [18]:
for file in images_ds.take(3):
  print(file.numpy())

b'images/dog/The 35 Cutest Dog Breeds That Will Make.jpg'
b'images/dog/20 Small Fluffy Dog Breeds - Best Small.jpg'
b'images/cat/lovely baby cat.jpg'


In [38]:
# Splitting into Train and test ds
train_size = int(image_count*0.8)

train_ds = images_ds.take(train_size)
test_ds = images_ds.skip(train_size)

len(train_ds), len(test_ds)

(12, 4)

In [27]:
# Extract label from file path
def get_label(file_path):
  import os
  parts = tf.strings.split(file_path, os.path.sep)
  return parts[-2]

get_label("images/dog/The 35 Cutest Dog Breeds That Will Make.jpg")

<tf.Tensor: shape=(), dtype=string, numpy=b'dog'>

In [39]:
# Process the images
def process_image(file_path):
  label = get_label(file_path)
  img = tf.io.read_file(file_path)
  img = tf.image.decode_jpeg(img)
  img = tf.image.resize(img, [128, 128])
  img = img/255.0
  return img, label

img, label = process_image("images/dog/The 35 Cutest Dog Breeds That Will Make.jpg")
img.numpy()[0][0], label.numpy()

(array([0.7607843, 0.7254902, 0.5665326], dtype=float32), b'dog')

In [40]:
# Process the train and test ds
train_ds = train_ds.map(process_image)
test_ds = test_ds.map(process_image)

In [44]:
for image, label in train_ds.take(1):
  print("Image:- ", image.numpy())
  print("Label:- ", label.numpy())

Image:-  [[[0.61563253 0.675609   0.49426708]
  [0.6735947  0.74525523 0.53493446]
  [0.7173218  0.8031262  0.55454963]
  ...
  [0.3521175  0.38741162 0.31909055]
  [0.42845398 0.4637481  0.38139516]
  [0.5264859  0.5538563  0.4758654 ]]

 [[0.6189451  0.65466017 0.5347346 ]
  [0.68087035 0.72785586 0.5820531 ]
  [0.72407144 0.79031503 0.6035091 ]
  ...
  [0.27398896 0.30928308 0.2410751 ]
  [0.37083662 0.40613073 0.3239244 ]
  [0.48723558 0.5189411  0.4400469 ]]

 [[0.6910763  0.6931443  0.6538367 ]
  [0.7236024  0.73589265 0.6724535 ]
  [0.7300209  0.7524803  0.65745187]
  ...
  [0.21496163 0.25025576 0.18276188]
  [0.30542752 0.34072164 0.259441  ]
  [0.42209065 0.45251346 0.37746748]]

 ...

 [[0.4197469  0.33094835 0.2789416 ]
  [0.40349475 0.31453946 0.26241964]
  [0.3933008  0.30434552 0.25151157]
  ...
  [0.28865182 0.15309496 0.04716533]
  [0.2937436  0.15433463 0.05619064]
  [0.530563   0.38591272 0.24678189]]

 [[0.14573886 0.08572046 0.07181115]
  [0.08701627 0.02551198 0.0

# tf dataset from text files


In [81]:
# Load text files
reviews_ds = tf.data.Dataset.list_files('reviews/*/*')
print(reviews_ds)
print(len(reviews_ds))

<_ShuffleDataset element_spec=TensorSpec(shape=(), dtype=tf.string, name=None)>
6


In [82]:
for file in reviews_ds.take(3):
  print(file.numpy())

b'reviews/positive/pos_2.txt'
b'reviews/negative/neg_2.txt'
b'reviews/negative/neg_1.txt'


In [83]:
# Extract label from file path
def get_label(file_path):
  import os
  parts = tf.strings.split(file_path, os.path.sep)
  return parts[-2]

get_label("reviews/negative/neg_1.txt")

<tf.Tensor: shape=(), dtype=string, numpy=b'negative'>

In [84]:
# Process the files
def process_file(file_path):
  label = get_label(file_path)
  review = tf.io.read_file(file_path)
  return review, label

review, label = process_file("reviews/negative/neg_1.txt")
review.numpy(), label.numpy()

(b"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.\n",
 b'negative')

In [85]:
reviews_ds = reviews_ds.map(process_file)

In [90]:
# Filter blank reviews
reviews_ds = reviews_ds.filter(lambda review, label: review!=" ")

In [91]:
for review, label in reviews_ds.as_numpy_iterator():
  print("Review:- ", review)
  print("Label:- ", label)

Review:-  b"Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And then we have Jake with his closet which totally ruins all the film! I expected to see a BOOGEYMAN similar movie, and instead i watched a drama with some meaningless thriller spots.<br /><br />3 out of 10 just for the well playing parents & descent dialogs. As for the shots with Jake: just ignore them.\n"
Label:-  b'negative'
Review:-  b"This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not really funny anymore

In [93]:
# Perform in one line
reviews_ds = tf.data.Dataset.list_files('reviews/*/*')
final_ds = reviews_ds.map(process_file).filter(lambda review, label: review!=" ")
for review, label in final_ds.as_numpy_iterator():
  print("Review:- ", review)
  print("Label:- ", label)

Review:-  b"This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not really funny anymore, and it's continued its decline further to the complete waste of time it is today.<br /><br />It's truly disgraceful how far this show has fallen. The writing is painfully bad, the performances are almost as bad - if not for the mildly entertaining respite of the guest-hosts, this show probably wouldn't still be on the air. I find it so hard to believe that the same creator that hand-selected the original cast also chose the band of hacks that followed. How can one recognize such brilliance and then see fit to replace it with such mediocrity? I felt I must give 2 stars out of respect for the original cast that made this show such a huge success. As it is now, the show is just awful. I can't believe it's still on the air.\n"
Label:-  b'negative'
Review:-  b"One of the other 